# Statistical traps that survive correct code

None of the code below has a bug. Every number is computed correctly, and every headline
is misleading. After you have checked the shifts and the merges, ask whether the *question*
the numbers answer is the one you think it is.

Each trap: the smallest table that shows it, then the same thing on the real data, then
what to say aloud.

**What's in here**
1. Simpson's paradox
2. survivorship bias
3. regression to the mean
4. selection on the outcome (best of N)
5. multiple comparisons
6. spurious correlation of trends (pointer)
7. base-rate neglect
8. aggregation traps: mean of ratios, unweighted averages, volume-weighted prices
9. collider / Berkson bias
10. Goodhart: optimise a metric, get the metric
11. the improvement that is one period
12. extrapolation outside the training range
13. questions to ask about any surprising result

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

hourly = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).set_index("time")
meters = pd.read_csv("../data/meters.csv", parse_dates=["signup_date"])
meters["region"] = meters["region"].str.title()
readings = pd.read_csv("../data/meter_readings_daily.csv", parse_dates=["date"])
readings = readings[readings["meter_id"].isin(meters["meter_id"])]
print("hourly", hourly.shape, "| meters", meters.shape, "| readings", readings.shape)

hourly (17520, 5) | meters (300, 7) | readings (107303, 3)


## 1. Simpson's paradox

Two regions, two customer types. Cost per kWh, with the counts written out:

In [2]:
toy = pd.DataFrame({
    "region":        ["A", "A", "B", "B"],
    "customer_type": ["residential", "sme", "residential", "sme"],
    "kwh":           [9000, 1000, 1000, 9000],
    "cost":          [2700, 200, 320, 1890],
})
toy["cost_per_kwh"] = toy["cost"] / toy["kwh"]
toy

,region,customer_type,kwh,cost,cost_per_kwh
0,A,residential,9000,2700,0.30
1,A,sme,1000,200,0.20
2,B,residential,1000,320,0.32
3,B,sme,9000,1890,0.21


Within each customer type, **region A is cheaper**: residential 0.30 vs 0.32, SME 0.20 vs 0.21.
Now the overall cost per kWh by region:

In [3]:
by_region = toy.groupby("region")[["cost", "kwh"]].sum()
by_region["cost_per_kwh"] = by_region["cost"] / by_region["kwh"]
by_region

,cost,kwh,cost_per_kwh
region,,,
A,2900,10000,0.290
B,2210,10000,0.221


Overall, **region A is more expensive** (0.29 vs 0.22), even though it is cheaper for
residential customers *and* cheaper for SMEs. Look at the mix:

In [4]:
mix = toy.pivot_table(index="region", columns="customer_type", values="kwh", aggfunc="sum")
mix["sme_share"] = mix["sme"] / (mix["sme"] + mix["residential"])
mix

customer_type,residential,sme,sme_share
region,,,
A,9000,1000,0.1
B,1000,9000,0.9


Region A's kWh are 90% residential (the expensive type); region B's are 90% SME (the cheap
type). The overall numbers are driven by *mix*, not by tariffs. Swap the mix and the overall
ranking flips, while the within-type ranking never changes. That is the paradox: the
aggregate can say anything the mix makes it say.

The only safe comparison is **within** customer type. On the real meters (simulated unit
rates, real regional mix):

In [5]:
annual = readings.groupby("meter_id")["kwh"].sum().rename("kwh")
m = meters.merge(annual, on="meter_id")
rate = {"residential": 0.30, "sme": 0.18}
m["rate"] = m["customer_type"].map(rate)
m.loc[m["region"] == "London", "rate"] = m.loc[m["region"] == "London", "rate"] + 0.02   # London dearer within each type
m["cost"] = m["kwh"] * m["rate"]
m[["meter_id", "region", "customer_type", "kwh", "rate", "cost"]].head(4)

,meter_id,region,customer_type,kwh,rate,cost
0,M100000,London,sme,21939.204,0.20,4387.84080
1,M100001,London,residential,2336.996,0.32,747.83872
2,M100002,London,residential,3639.039,0.32,1164.49248
3,M100003,Scotland,residential,2567.712,0.30,770.31360


In [6]:
sums = m.groupby("region")[["cost", "kwh"]].sum()
overall = (sums["cost"] / sums["kwh"]).rename("overall cost_per_kwh")
overall.round(3).sort_values()

region
Midlands    0.231
Wales       0.232
North       0.237
London      0.260
Scotland    0.262
Name: overall cost_per_kwh, dtype: float64

In [7]:
within_cost = m.pivot_table(index="region", columns="customer_type", values="cost", aggfunc="sum")
within_kwh = m.pivot_table(index="region", columns="customer_type", values="kwh", aggfunc="sum")
within = (within_cost / within_kwh).round(3)
within["sme_kwh_share"] = (within_kwh["sme"] / within_kwh.sum(axis=1)).round(2)
within

customer_type,residential,sme,sme_kwh_share
region,,,
London,0.32,0.20,0.50
Midlands,0.30,0.18,0.57
North,0.30,0.18,0.52
Scotland,0.30,0.18,0.32
Wales,0.30,0.18,0.57


Within each type every region pays the same except London (dearer). Yet Scotland is the
most expensive overall: its SME share (the cheap kWh) is the lowest. The overall ranking is
a statement about customer mix.

**What to say aloud:** "Before I compare regions I want to know whether they have the same
customer mix. If not, compare within type or standardise the mix."

## 2. Survivorship bias

Five customers. Their January and December usage, and whether they churned before December:

In [8]:
toy = pd.DataFrame({
    "customer": ["c1", "c2", "c3", "c4", "c5"],
    "jan":      [10, 10, 10, 10, 10],
    "dec":      [11, 10, 11,  9,  4],
    "churned":  [False, False, False, False, True],
})
toy["growth"] = toy["dec"] / toy["jan"] - 1
toy

,customer,jan,dec,churned,growth
0,c1,10,11,False,0.1
1,c2,10,10,False,0.0
2,c3,10,11,False,0.1
3,c4,10,9,False,-0.1
4,c5,10,4,True,-0.6


In [9]:
print("true average growth, all five  :", round(toy["growth"].mean(), 3))
survivors = toy[~toy["churned"]]
print("measured growth, survivors only:", round(survivors["growth"].mean(), 3))

true average growth, all five  : -0.1
measured growth, survivors only: 0.025


The customer who churned was the one whose usage collapsed. Measure only on the customers
still present in December and growth looks positive when the truth is negative.

Simulated at scale (2,000 customers; churn probability rises as usage falls):

In [10]:
n = 2000
base = rng.lognormal(np.log(9), 0.4, n)                # kWh/day in January
growth = rng.normal(0.0, 0.08, n)                      # true growth: mean 0
dec_all = base * (1 + growth)
churn_prob = 1 / (1 + np.exp(8 * growth + 1.5))        # falling usage -> more likely to churn
churned = rng.random(n) < churn_prob
print("churn rate:", round(churned.mean(), 3))

churn rate: 0.202


In [11]:
survivors = ~churned
print("true mean growth, everyone      :", round(growth.mean(), 4))
print("measured growth, survivors only :", round(dec_all[survivors].mean() / base[survivors].mean() - 1, 4))
print("mean growth of the churned      :", round(growth[churned].mean(), 4))

true mean growth, everyone      : 0.0013
measured growth, survivors only : 0.0119
mean growth of the churned      : -0.0382


**What to say aloud:** "Who is missing from this sample, and could the reason they are
missing be related to the thing I am measuring?"

## 3. Regression to the mean

Six meters, two days. Each meter's true level is 10; each day adds noise.

In [12]:
toy = pd.DataFrame({
    "meter": ["m1", "m2", "m3", "m4", "m5", "m6"],
    "day1":  [13, 8, 12, 9, 7, 11],
    "day2":  [10, 11, 9, 12, 8, 10],
})
toy

,meter,day1,day2
0,m1,13,10
1,m2,8,11
2,m3,12,9
3,m4,9,12
4,m5,7,8
5,m6,11,10


In [13]:
top2 = toy.nlargest(2, "day1")
print("top 2 on day 1:")
print(top2)
print()
print("their day-1 mean:", top2["day1"].mean(), " -> day-2 mean:", top2["day2"].mean())
print("everyone's day-1 mean:", toy["day1"].mean(), " -> day-2 mean:", toy["day2"].mean())

top 2 on day 1:
  meter  day1  day2
0    m1    13    10
2    m3    12     9

their day-1 mean: 12.5  -> day-2 mean: 9.5
everyone's day-1 mean: 10.0  -> day-2 mean: 10.0


The top group fell from 12.5 to 9.5 with no intervention. They were "top" partly because
day 1 was noisy in their favour, and noise does not repeat.

On the real daily readings: the 20 heaviest meters on one day, and the same meters the next day.

In [14]:
wide = readings.pivot(index="meter_id", columns="date", values="kwh")
d0 = pd.Timestamp("2023-01-16")
d1 = pd.Timestamp("2023-01-17")
both = wide[[d0, d1]].dropna()
both.columns = ["day0", "day1"]
both.head(3)

,day0,day1
meter_id,,
M100001,6.507,9.332
M100002,11.862,11.000
M100003,6.373,11.427


In [15]:
top = both.nlargest(20, "day0")
print("population: ", round(both["day0"].mean(), 2), "->", round(both["day1"].mean(), 2))
print("top-20     : ", round(top["day0"].mean(), 2), "->", round(top["day1"].mean(), 2),
      "  change", round(top["day1"].mean() / top["day0"].mean() - 1, 3))

population:  20.89 -> 20.83
top-20     :  116.61 -> 105.9   change -0.092


Selecting on "unusually high relative to their own average" makes the fall even bigger:

In [16]:
own_mean = wide.loc[:, wide.columns.month == 1].mean(axis=1)
both["ratio_to_own_mean"] = both["day0"] / own_mean.reindex(both.index)
top_rel = both.nlargest(20, "ratio_to_own_mean")
print("top-20 by ratio:", round(top_rel["day0"].mean(), 2), "->", round(top_rel["day1"].mean(), 2),
      "  change", round(top_rel["day1"].mean() / top_rel["day0"].mean() - 1, 3))

top-20 by ratio: 45.14 -> 28.09   change -0.378


A campaign "targeting the heaviest users" that reports a 10% reduction has to be compared
with a control group selected the same way.

**What to say aloud:** "Was this group selected on the same variable we are now measuring
the change in?"

## 4. Selection on the outcome: best of N

Five random coin-flip "strategies" over 10 days each. Pick the best one on those 10 days,
then look at its next 10 days.

In [17]:
rng_toy = np.random.default_rng(2)
flips = rng_toy.choice([-1, 1], size=(5, 20))        # 5 strategies, 20 days of ±1 P&L
first10 = flips[:, :10].sum(axis=1)
next10 = flips[:, 10:].sum(axis=1)
toy = pd.DataFrame({"strategy": ["s1", "s2", "s3", "s4", "s5"], "pnl first 10 days": first10, "pnl next 10 days": next10})
toy

,strategy,pnl first 10 days,pnl next 10 days
0,s1,-4,2
1,s2,0,0
2,s3,4,0
3,s4,4,0
4,s5,-2,-2


In [18]:
best = toy["pnl first 10 days"].idxmax()
print("best on the first 10 days:", toy.loc[best, "strategy"], "with", toy.loc[best, "pnl first 10 days"])
print("the same strategy on the next 10 days:", toy.loc[best, "pnl next 10 days"])

best on the first 10 days: s3 with 4
the same strategy on the next 10 days: 0


Pure luck picked the winner; its future is a coin flip. The expected maximum of N noise draws
grows with N: about +1.9 standard errors for N = 20. On the real prices with 20 random rules:

In [19]:
dp = hourly["price_eur_mwh"].diff().dropna()
y22 = dp.loc["2022"].values
y23 = dp.loc["2023"].values
n_rules = 20
rules = rng.choice([-1, 1], size=(n_rules, len(dp)))    # random signals, no information

sharpe22 = []
sharpe23 = []
for i in range(n_rules):
    pnl22 = rules[i, :len(y22)] * y22
    pnl23 = rules[i, len(y22):] * y23
    sharpe22.append(pnl22.mean() / pnl22.std() * np.sqrt(8760))
    sharpe23.append(pnl23.mean() / pnl23.std() * np.sqrt(8760))
sharpe22 = np.array(sharpe22)
sharpe23 = np.array(sharpe23)
best = sharpe22.argmax()
print("best of 20 rules on 2022: Sharpe", round(sharpe22[best], 2), "| same rule on 2023:", round(sharpe23[best], 2))
print("expected max of 20 standard normal draws:", round(stats.norm.ppf(1 - 1 / 21), 2), "standard errors")

best of 20 rules on 2022: Sharpe 2.15 | same rule on 2023: -0.67
expected max of 20 standard normal draws: 1.67 standard errors


**What to say aloud:** "How many things were tried before this one was chosen, and was the
number I am shown computed on the data used to choose it?"

## 5. Multiple comparisons

Test 20 pure-noise features against a noise target. At the 5% level you expect one false
positive by construction. Print all 20 p-values:

In [20]:
rng_toy = np.random.default_rng(1)
n = 500
target = rng_toy.normal(size=n)
p_values = []
for j in range(20):
    feature = rng_toy.normal(size=n)
    r, p = stats.pearsonr(feature, target)
    p_values.append(p)
p_values = pd.Series(p_values, index=["f" + str(j) for j in range(20)]).round(3)
print(p_values.to_string())
print()
print("features with p < 0.05:", (p_values < 0.05).sum(), "of 20")

f0     0.992
f1     0.462
f2     0.192
f3     0.438
f4     0.911
f5     0.637
f6     0.751
f7     0.044
f8     0.208
f9     0.226
f10    0.344
f11    0.222
f12    0.857
f13    0.120
f14    0.102
f15    0.000
f16    0.413
f17    0.474
f18    0.505
f19    0.847

features with p < 0.05: 2 of 20


Two features came out below 0.05. Both are noise: with 20 tests at the 5% level, one false
positive is the *expected* outcome, two is unremarkable. The mechanics of Bonferroni and Benjamini–Hochberg are
in `03_scipy/04_hypothesis_tests_power_and_pitfalls.ipynb`. On the real data, 50 noise
features regressed on 2023 consumption:

In [21]:
y = hourly["consumption_mwh"].loc["2023"].values
y = y - y.mean()
Z = rng.normal(size=(len(y), 50))
X = np.column_stack([np.ones(len(y)), Z])
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
e = y - X @ beta
se = np.sqrt((e @ e) / (len(y) - X.shape[1]) * np.diag(np.linalg.inv(X.T @ X)))
t = beta / se
p = 2 * (1 - stats.t.cdf(np.abs(t[1:]), df=len(y) - X.shape[1]))
print("noise features with p < 0.05:", (p < 0.05).sum(), "of 50 | smallest p:", round(p.min(), 4))
print("R² of the noise-only model:", round(1 - (e @ e) / (y @ y), 4))

noise features with p < 0.05: 3 of 50 | smallest p: 0.0219
R² of the noise-only model: 0.0057


## 6. Spurious correlation of trends (pointer)

Two unrelated trending series correlate strongly in levels and not at all in differences.
Shown step by step in `04_classical_time_series_stats.ipynb`, section 8. Any 2022
correlation with price must be re-checked in changes or after detrending.

## 7. Base-rate neglect

100 hours. 2 of them are real spikes. A detector catches 95% of spikes and wrongly flags 5%
of quiet hours. Count what it flags:

In [22]:
spikes = 2
quiet = 98
true_alarms = 0.95 * spikes
false_alarms = 0.05 * quiet
print("true alarms :", true_alarms)
print("false alarms:", false_alarms)
print("precision = true / (true + false) =", round(true_alarms / (true_alarms + false_alarms), 3))

true alarms : 1.9
false alarms: 4.9
precision = true / (true + false) = 0.279


Fewer than a third of alarms are real, from a "95% accurate" detector. Precision depends on
the base rate:

In [23]:
rows = []
for base_rate in [0.005, 0.02, 0.10, 0.30]:
    tp = 0.95 * base_rate
    fp = 0.05 * (1 - base_rate)
    rows.append({"base_rate": base_rate, "precision": round(tp / (tp + fp), 3)})
pd.DataFrame(rows)

,base_rate,precision
0,0.005,0.087
1,0.020,0.279
2,0.100,0.679
3,0.300,0.891


**What to say aloud:** "What is the base rate, and what is precision at the threshold we
would actually use?"

## 8. Aggregation traps

### Mean of ratios vs ratio of sums

Three hours. Cost per MWh paid in each hour, and overall:

In [24]:
toy = pd.DataFrame({"hour": [1, 2, 3], "price": [50, 100, 300], "volume_mwh": [10, 10, 1]})
toy["cost"] = toy["price"] * toy["volume_mwh"]
toy["cost_per_mwh"] = toy["cost"] / toy["volume_mwh"]
toy

,hour,price,volume_mwh,cost,cost_per_mwh
0,1,50,10,500,50.0
1,2,100,10,1000,100.0
2,3,300,1,300,300.0


In [25]:
print("mean of the hourly prices     :", round(toy["price"].mean(), 2))
print("total cost / total volume     :", round(toy["cost"].sum() / toy["volume_mwh"].sum(), 2))

mean of the hourly prices     : 150.0
total cost / total volume     : 85.71


The expensive hour had almost no volume, so the price actually *paid* is far below the
simple mean. On the real data for a retailer serving 1% of load:

In [26]:
h = hourly.loc["2023"]
volume = h["consumption_mwh"] * 0.01
cost = h["price_eur_mwh"] * volume
print("time-weighted mean price     :", round(h["price_eur_mwh"].mean(), 2))
print("volume-weighted (paid) price :", round(cost.sum() / volume.sum(), 2))

time-weighted mean price     : 84.76
volume-weighted (paid) price : 88.38


### Unweighted average of percentages

Three regions with very different sizes:

In [27]:
g = pd.DataFrame({"region": ["A", "B", "C"], "meters": [900, 50, 50], "solar_share": [0.05, 0.40, 0.45]})
g["meters_with_solar"] = g["meters"] * g["solar_share"]
g

,region,meters,solar_share,meters_with_solar
0,A,900,0.05,45.0
1,B,50,0.40,20.0
2,C,50,0.45,22.5


In [28]:
print("unweighted mean of the three shares:", round(g["solar_share"].mean(), 3))
print("actual share of all meters         :", round(g["meters_with_solar"].sum() / g["meters"].sum(), 3))

unweighted mean of the three shares: 0.3
actual share of all meters         : 0.088


**What to say aloud:** "Is this a mean of ratios or a ratio of sums, and which one does the
business question need?"

## 9. Collider / Berkson bias

Bill size and service quality are independent. Customers complain when the bill is high
**or** the service is poor. Look only at complainers and a relationship appears.
Eight customers:

In [29]:
toy = pd.DataFrame({
    "customer":      ["c1", "c2", "c3", "c4", "c5", "c6", "c7", "c8"],
    "high_bill":     [1, 1, 1, 1, 0, 0, 0, 0],
    "poor_service":  [1, 1, 0, 0, 1, 1, 0, 0],
})
toy["complained"] = (toy["high_bill"] == 1) | (toy["poor_service"] == 1)
toy

,customer,high_bill,poor_service,complained
0,c1,1,1,True
1,c2,1,1,True
2,c3,1,0,True
3,c4,1,0,True
4,c5,0,1,True
5,c6,0,1,True
6,c7,0,0,False
7,c8,0,0,False


In [30]:
print("everyone     : corr(high_bill, poor_service) =", round(toy["high_bill"].corr(toy["poor_service"]), 3))
c = toy[toy["complained"]]
print("complainers  : corr(high_bill, poor_service) =", round(c["high_bill"].corr(c["poor_service"]), 3))

everyone     : corr(high_bill, poor_service) = 0.0
complainers  : corr(high_bill, poor_service) = -0.5


Among complainers, the ones with high bills tend to have *good* service (they complained
about the bill, not the service). An analyst who only sees the complaints file concludes
"big customers get better treatment". Simulated at scale:

In [31]:
n = 20000
bill = rng.normal(size=n)
poor_service = rng.normal(size=n)
complained = (bill > 1) | (poor_service > 1)
print("population  :", round(np.corrcoef(bill, poor_service)[0, 1], 3))
print("complainers :", round(np.corrcoef(bill[complained], poor_service[complained])[0, 1], 3), " n =", complained.sum())

population  : 0.01
complainers : -0.561  n = 5752


**What to say aloud:** "Is the sample I'm analysing defined by something that both
variables influence?"

## 10. Goodhart: optimise a metric, get the metric

### MAPE rewards forecasting low

One actual value of 100. Compare a forecast of half (50) with a forecast of double (200):

In [32]:
actual = 100.0
for forecast in [50.0, 200.0]:
    ape = abs(actual - forecast) / actual
    print("forecast", forecast, "-> absolute percentage error", ape)

forecast 50.0 -> absolute percentage error 0.5
forecast 200.0 -> absolute percentage error 1.0


Under-forecasting can never cost more than 100%; over-forecasting has no ceiling. So a
forecaster judged on MAPE is rewarded for aiming low. The MAPE-optimal constant for 2023
prices sits well below the mean:

In [33]:
from scipy.optimize import minimize_scalar

y = hourly["price_eur_mwh"].loc["2023"]
y = y[y > 0].values                                   # MAPE needs a positive denominator

def mape_of_constant(c):
    return np.mean(np.abs(y - c) / y)

def rmse_of_constant(c):
    return np.sqrt(np.mean((y - c) ** 2))

best_mape = minimize_scalar(mape_of_constant, bounds=(y.min(), y.max()), method="bounded").x
best_rmse = minimize_scalar(rmse_of_constant, bounds=(y.min(), y.max()), method="bounded").x
print("constant that minimises MAPE:", round(best_mape, 1))
print("constant that minimises RMSE:", round(best_rmse, 1), "(the mean)")
print("bias of the MAPE choice     :", round(best_mape / y.mean() - 1, 3))

constant that minimises MAPE: 62.4
constant that minimises RMSE: 85.1 (the mean)
bias of the MAPE choice     : -0.267


### Directional accuracy ignores size

Six hours. A forecaster gets the sign right on the five small moves and wrong on the one big move:

In [34]:
toy = pd.DataFrame({"move": [1, -1, 2, -1, 1, -20]})
toy["signal"] = [1, -1, 1, -1, 1, 1]               # right five times, wrong on the big one
toy["pnl"] = toy["signal"] * toy["move"]
toy

,move,signal,pnl
0,1,1,1
1,-1,-1,1
2,2,1,2
3,-1,-1,1
4,1,1,1
5,-20,1,-20


In [35]:
print("directional accuracy:", round((np.sign(toy["signal"]) == np.sign(toy["move"])).mean(), 3))
print("total P&L           :", toy["pnl"].sum())

directional accuracy: 0.833
total P&L           : -14


**What to say aloud:** "Which decision does this metric stand in for, and can the metric be
improved without improving the decision?"

## 11. The improvement that is one period

Model B beats A over the year. Per month, A wins almost always; B's edge is one month.

In [36]:
months = pd.period_range("2023-01", "2023-12", freq="M")
rmse_a = pd.Series(rng.normal(100, 5, 12).round(1), index=months)
rmse_b = rmse_a + rng.normal(3, 1, 12).round(1)              # B is worse in a typical month
rmse_b.iloc[6] = rmse_a.iloc[6] - 45                         # ... except one month
tbl = pd.DataFrame({"rmse_A": rmse_a, "rmse_B": rmse_b})
tbl.T

,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12
rmse_A,106.6,102.3,96.9,105.9,102.9,97.4,100.2,104.9,101.6,104.6,97.3,96.3
rmse_B,109.9,104.0,98.8,107.6,105.1,98.7,55.2,107.8,105.1,108.3,99.8,100.0


In [37]:
print("annual RMSE A:", round(np.sqrt((rmse_a ** 2).mean()), 1))
print("annual RMSE B:", round(np.sqrt((rmse_b ** 2).mean()), 1), " -> B 'wins'")
print("months where A is better:", (rmse_a < rmse_b).sum(), "of 12")

annual RMSE A: 101.5
annual RMSE B: 101.0  -> B 'wins'
months where A is better: 11 of 12


**What to say aloud:** "Show me the per-period wins, not just the total. Is the gain
concentrated, and do I expect that regime to recur?"

## 12. Extrapolation outside the training range

Fit demand on temperature using 2022 (max temperature seen ≈ 28 °C), then ask both a cubic
and a piecewise-linear model what happens at 35 °C.

In [38]:
h22 = hourly.loc["2022"]
t = h22["temp_c"].values
y = h22["consumption_mwh"].values
print("training temperature range:", round(t.min(), 1), "to", round(t.max(), 1), "°C")

training temperature range:

 -6.2 to 27.7 °C


In [39]:
X_cubic = np.column_stack([np.ones_like(t), t, t ** 2, t ** 3])
b_cubic, *_ = np.linalg.lstsq(X_cubic, y, rcond=None)

hdd = np.clip(15 - t, 0, None)
cdd = np.clip(t - 22, 0, None)
X_piece = np.column_stack([np.ones_like(t), hdd, cdd])
b_piece, *_ = np.linalg.lstsq(X_piece, y, rcond=None)

rows = []
for T in [25, 30, 35, 40]:
    cubic = b_cubic @ [1, T, T ** 2, T ** 3]
    piece = b_piece @ [1, max(15 - T, 0), max(T - 22, 0)]
    rows.append({"temp_c": T, "cubic": round(cubic), "piecewise": round(piece)})
print("mean load in 2022:", round(y.mean()))
pd.DataFrame(rows)

mean load in 2022: 29407


,temp_c,cubic,piecewise
0,25,33343,29645
1,30,43990,32217
2,35,62283,34790
3,40,90043,37363


Neither model has seen 35 °C. The cubic's answer is nonsense; the piecewise model at least
continues its last slope. Both are assumptions, not estimates.

**What to say aloud:** "Where is this input relative to the training range? If it is
outside, the model's answer is an assumption."

## 13. Questions to ask about any surprising result

1. What is one observation, and how many are there really (after duplicates, after autocorrelation)?
2. Who or what is missing from the sample, and is the reason related to the outcome?
3. Was this group selected on the variable I am now measuring?
4. How many alternatives were tried before this one was reported?
5. Was the number computed on the data used to choose the model?
6. Do the subgroups agree with the total? If not, is the mix different?
7. Is this a mean of ratios or a ratio of sums? Weighted by what?
8. Does the comparison hold period by period, or is it one regime?
9. What is the base rate, and what is precision at the operating threshold?
10. Is the input inside the training range?
11. Is the metric the decision, or a proxy that can be gamed?
12. What would I expect to see if there were no effect at all, and is this distinguishable from it?